# PatchTST

In [ ]:
%pip install -q mlflow neuralforecast scikit-learn pandas matplotlib

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    pass
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.losses.pytorch import MAE
from src.dl import build_series_matrix, HORIZON
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

from src.data import load_raw, submission_ids
from src.features import WalmartFeatureBuilder, MARKDOWN_COLS
from src.metrics import wmae, holiday_weights
from src.validation import year_back_split, time_folds, DEV_TRAIN_END, TRAIN_END
from src.tracking import setup_mlflow, REGISTERED_MODEL_NAME

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("MLflow tracking:", setup_mlflow("PatchTST_Training"))

In [ ]:
raw = load_raw()
train, test = raw["train"], raw["test"]

tr_idx, va_idx = year_back_split(train["Date"])
dev_train, dev_valid = train.iloc[tr_idx], train.iloc[va_idx]
print(f"train: {train.shape}, test: {test.shape}")
print(f"dev_train: {len(dev_train):,} რიგი (<= {DEV_TRAIN_END.date()}), "
      f"dev_valid: {len(dev_valid):,} რიგი (39 კვირა, შეიცავს Thanksgiving/Christmas/Super Bowl-ს)")

In [ ]:
values, mask, keys, dates = build_series_matrix(train)
long = pd.DataFrame(values,
                    index=pd.MultiIndex.from_frame(keys),
                    columns=dates).stack().rename("y").reset_index()
long.columns = ["Store", "Dept", "ds", "y"]
long["unique_id"] = long["Store"].astype(str) + "_" + long["Dept"].astype(str)
long = long[["unique_id", "ds", "y"]]

with mlflow.start_run(run_name="PatchTST_Preprocessing"):
    mlflow.log_params({"format": "long (unique_id, ds, y)", "grid": "W-FRI",
                       "missing_policy": "fill 0 (no mask support)",
                       "scaling": "robust (per-window, in-model)"})
    mlflow.log_metrics({"n_series": long["unique_id"].nunique(),
                        "n_rows": len(long)})

dev_long = long[long["ds"] <= DEV_TRAIN_END]
print(f"long: {len(long)} რიგი | dev: {len(dev_long)}")

In [ ]:
def make_model(**kw):
    p = dict(h=HORIZON, input_size=104, patch_len=8, stride=4,
             hidden_size=128, n_heads=4, loss=MAE(), scaler_type="robust",
             max_steps=1500, batch_size=64, learning_rate=1e-3,
             random_seed=RANDOM_STATE, start_padding_enabled=True)
    p.update(kw)
    return PatchTST(**p)

def eval_forecast(fc_df, frame, col="PatchTST"):
    f = frame.copy()
    f["unique_id"] = f["Store"].astype(str) + "_" + f["Dept"].astype(str)
    m = f.merge(fc_df, left_on=["unique_id", "Date"],
                right_on=["unique_id", "ds"], how="left")
    return wmae(m["Weekly_Sales"], m[col].fillna(0.0), m["IsHoliday"])

configs = [dict(patch_len=8,  stride=4, hidden_size=128),
           dict(patch_len=16, stride=8, hidden_size=128),
           dict(patch_len=8,  stride=4, hidden_size=256)]

best_wmae, best_cfg = None, None
with mlflow.start_run(run_name="PatchTST_CV"):
    for cfg in configs:
        name = f"p{cfg['patch_len']}_s{cfg['stride']}_h{cfg['hidden_size']}"
        with mlflow.start_run(run_name=name, nested=True):
            nf = NeuralForecast(models=[make_model(**cfg)], freq="W-FRI")
            nf.fit(dev_long)
            fc = nf.predict()
            if "unique_id" not in fc.columns:
                fc = fc.reset_index()
            score = eval_forecast(fc, dev_valid)
            mlflow.log_params(cfg)
            mlflow.log_metric("valid_wmae", score)
            print(f"{name}: valid WMAE {score:.0f}")
            if best_wmae is None or score < best_wmae:
                best_wmae, best_cfg = score, cfg

print(f"\nსაუკეთესო: {best_cfg} (WMAE {best_wmae:.0f})")

In [ ]:
class ForecastTablePyfunc(mlflow.pyfunc.PythonModel):

    def __init__(self, table):
        self.table = table
    def predict(self, context, model_input, params=None):
        X = model_input[["Store", "Dept", "Date"]].copy()
        X["Date"] = pd.to_datetime(X["Date"])
        m = X.merge(self.table, on=["Store", "Dept", "Date"], how="left")
        return m["yhat"].fillna(0.0).to_numpy()

with mlflow.start_run(run_name="PatchTST_Final") as run:
    nf = NeuralForecast(models=[make_model(**best_cfg)], freq="W-FRI")
    nf.fit(long)
    fc = nf.predict()
    if "unique_id" not in fc.columns:
        fc = fc.reset_index()

    parts = fc["unique_id"].str.split("_", expand=True).astype(int)
    table = pd.DataFrame({"Store": parts[0], "Dept": parts[1],
                          "Date": pd.to_datetime(fc["ds"]),
                          "yhat": fc["PatchTST"].to_numpy()})

    assert set(pd.to_datetime(test["Date"].unique())) <= set(table["Date"].unique())

    pyfunc_model = ForecastTablePyfunc(table)
    mlflow.log_params(best_cfg)
    mlflow.log_metric("valid_wmae", best_wmae)
    mlflow.pyfunc.log_model(name="model", python_model=pyfunc_model)
    patchtst_final_run_id = run.info.run_id

print("run:", patchtst_final_run_id)

In [ ]:
REGISTER_AS_CHAMPION = False

if REGISTER_AS_CHAMPION:
    from mlflow import MlflowClient
    mv = mlflow.register_model(f"runs:/{patchtst_final_run_id}/model", REGISTERED_MODEL_NAME)
    MlflowClient().set_registered_model_alias(REGISTERED_MODEL_NAME, "champion", mv.version)
    print(f"დარეგისტრირდა: {REGISTERED_MODEL_NAME} v{mv.version} (alias: champion)")